In [ ]:
# Install EasyDiffraction and prepare the example when running in Colab.
import importlib.util
import io
import os
import subprocess
import sys
import tempfile
import urllib.request
import zipfile
from pathlib import Path

if importlib.util.find_spec("edi") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--only-binary=:all:",
        "--index-url", "https://enhantica.github.io/edi/pypi/simple/",
        "easydiffraction",
    ])

if "google.colab" in sys.modules:
    workspace = Path(tempfile.mkdtemp(prefix="edi-tutorial-"))
    (workspace / "notebooks").mkdir()
    os.chdir(workspace / "notebooks")
    target = Path('../cli/pd-xray-cwl_nacl_simulation/project').parent.resolve()
    target.mkdir(parents=True, exist_ok=True)
    url = "https://enhantica.github.io/edi/assets/tutorials/pd-xray-cwl_nacl_simulation.zip"
    with urllib.request.urlopen(url, timeout=60) as response:
        data = response.read()
    with zipfile.ZipFile(io.BytesIO(data)) as bundle:
        for member in bundle.infolist():
            if not (target / member.filename).resolve().is_relative_to(target):
                raise ValueError("Unexpected path in tutorial bundle")
        bundle.extractall(target)


In [ ]:
"""Calculation without data: NaCl, laboratory X-ray."""

# Calculation without data: NaCl, laboratory X-ray

Calculate a laboratory X-ray pattern of rock salt with Cu K&alpha;₁ radiation and a
monochromator's polarization correction, without any measured data.

The project is the catalogue case `pd-xray-cwl_nacl_simulation`: the files the command line fits
with `python -m edi fit docs/user/cli/pd-xray-cwl_nacl_simulation/project` and the app opens from
its Examples.

> This tutorial is not run when the documentation is built yet. It loads its project from the edi
> repository, so it runs once a tutorial runner can provide the project elsewhere
> (`execution-exclusions.yml`).

In [ ]:
import edi

## Load the project

In [ ]:
project = edi.Project.load('../cli/pd-xray-cwl_nacl_simulation/project')
print(project.metadata.title)
print('Structures:', ', '.join(structure.name for structure in project.structures))
print('Experiments:', ', '.join(experiment.name for experiment in project.experiments))
print('Fitting mode:', project.fitting_mode)

## Calculate

There is nothing to fit: edi calculates the pattern from the model as it is.

In [ ]:
project.calculate()
pattern = project.experiments[0].data
print(len(pattern.intensity_calc), 'points, highest', round(max(pattern.intensity_calc), 1))

## Patterns and report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structure and the fit. Open it in a browser.

In [ ]:
project.report.save_html('pd-xray-cwl_nacl_simulation.html')

## Save

`save_as` writes the project with its new values to a folder of its own; the catalogue's copy
stays as it was.

In [ ]:
project.save_as('pd-xray-cwl_nacl_simulation')